In [7]:
import pandas as pd
import numpy as np
import utils
%reload_ext autoreload
%autoreload 2

df_raw_origin = utils.extract("by_place_of_origin.xls", sheet_name="PROVINCE")
df_raw_origin


,PROVINCE,1988,1989,1990,1991,1992,1993,1994,1995,1996,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,TOTAL
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Region I - Ilocos Region,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ILOCOS NORTE,1330,1544,1945,1795,1860,1975,1832,1992,2335,...,2416.0,2389.0,2380.0,2730.0,2605.0,2425.0,2221.0,1878.0,420.0,66055
3,ILOCOS SUR,785,823,1028,1018,990,1180,1145,1028,1271,...,1337.0,1322.0,1408.0,1575.0,1560.0,1525.0,1181.0,920.0,242.0,36538
4,LA UNION,864,789,847,977,1022,959,1001,898,987,...,1231.0,948.0,1265.0,1559.0,1255.0,1318.0,1115.0,950.0,215.0,32422
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,(a) - Isabela City (Region IX) and Cotabato Ci...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
178,"(b) - Republic Act No. 9355, ratified through ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
179,"(c) - Muslim Mindanao Act No. 201, ratified th...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
180,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [58]:
# clean the data

df_clean_origin = utils.clean(df_raw_origin)


df_clean_origin["Region"] = df_clean_origin["PROVINCE"].apply(
    lambda x: x if pd.notna(x) and "region" in str(x).lower() else np.nan
)
df_clean_origin = df_clean_origin.iloc[:-4]
df_clean_origin["Region"] = df_clean_origin["Region"].ffill()
df_clean_origin = df_clean_origin[df_clean_origin["PROVINCE"] != df_clean_origin["Region"]].copy()
df_clean_origin.loc[df_clean_origin["PROVINCE"] == "Not Reported/No Response", "Region"] = "Not Reported"


df_clean_origin
                      

,PROVINCE,1988,1989,1990,1991,1992,1993,1994,1995,1996,...,2013,2014,2015,2016,2017,2018,2019,2020,TOTAL,Region
2,ILOCOS NORTE,1330,1544,1945,1795,1860,1975,1832,1992,2335,...,2389,2380,2730,2605,2425,2221,1878,420,66055,Region I - Ilocos Region
3,ILOCOS SUR,785,823,1028,1018,990,1180,1145,1028,1271,...,1322,1408,1575,1560,1525,1181,920,242,36538,Region I - Ilocos Region
4,LA UNION,864,789,847,977,1022,959,1001,898,987,...,948,1265,1559,1255,1318,1115,950,215,32422,Region I - Ilocos Region
5,PANGASINAN,2769,2581,3129,3080,3135,3378,3646,3160,3366,...,3663,3685,4690,4060,4003,3847,3083,686,107207,Region I - Ilocos Region
9,BATANES,2,0,0,1,0,2,3,2,1,...,9,13,12,6,9,5,4,0,140,Region II - Cagayan Valley
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,"PASAY CITY, (NCR FOURTH DISTRICT)",1045,1012,1130,1201,1134,1197,1131,1006,1091,...,785,750,661,687,585,485,437,110,28770,NATIONAL CAPITAL REGION
166,"PATEROS, (NCR FOURTH DISTRICT)",106,117,124,99,101,99,113,84,118,...,79,97,85,95,84,62,54,14,2975,NATIONAL CAPITAL REGION
167,"TAGUIG, (NCR FOURTH DISTRICT)",177,224,292,280,251,366,309,329,355,...,835,883,990,881,855,768,752,174,17640,NATIONAL CAPITAL REGION
168,NCR FOURTH DISTRICT,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4,NATIONAL CAPITAL REGION


In [59]:
# build dimension table
unique_locations = (
    df_clean_origin[["PROVINCE", "Region"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
unique_locations["Location_ID"] = [f"PH_LOC_{i+1:03d}" for i in unique_locations.index]
dim_ph_location = unique_locations[["Location_ID", "PROVINCE", "Region"]].rename(columns={"PROVINCE": "Province"})

dim_ph_location

,Location_ID,Province,Region
0,PH_LOC_001,ILOCOS NORTE,Region I - Ilocos Region
1,PH_LOC_002,ILOCOS SUR,Region I - Ilocos Region
2,PH_LOC_003,LA UNION,Region I - Ilocos Region
3,PH_LOC_004,PANGASINAN,Region I - Ilocos Region
4,PH_LOC_005,BATANES,Region II - Cagayan Valley
...,...,...,...
112,PH_LOC_113,"PASAY CITY, (NCR FOURTH DISTRICT)",NATIONAL CAPITAL REGION
113,PH_LOC_114,"PATEROS, (NCR FOURTH DISTRICT)",NATIONAL CAPITAL REGION
114,PH_LOC_115,"TAGUIG, (NCR FOURTH DISTRICT)",NATIONAL CAPITAL REGION
115,PH_LOC_116,NCR FOURTH DISTRICT,NATIONAL CAPITAL REGION


In [60]:
# make fact table
df_fact_prep = df_clean_origin.merge(
    dim_ph_location, left_on="PROVINCE", right_on="Province", how="left"
)

year_cols = [col for col in df_clean_origin.columns if col not in ["PROVINCE", "Region", "TOTAL"]]
fact_ph_origin = pd.melt(
    df_fact_prep,
    id_vars=["Location_ID"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Emigrant_Count"
)

fact_ph_origin["Emigrant_Count"] = fact_ph_origin["Emigrant_Count"].astype(int)
fact_ph_origin

,Location_ID,Year,Emigrant_Count
0,PH_LOC_001,1988,1330
1,PH_LOC_002,1988,785
2,PH_LOC_003,1988,864
3,PH_LOC_004,1988,2769
4,PH_LOC_005,1988,2
...,...,...,...
3856,PH_LOC_113,2020,110
3857,PH_LOC_114,2020,14
3858,PH_LOC_115,2020,174
3859,PH_LOC_116,2020,0


In [ ]:
# download csv file
utils.load(dim_ph_location, "dim_ph_location.csv")
utils.load(fact_ph_origin, "fact_ph_origin.csv")